In [0]:
# ============================================================
# BRONZE TO SILVER: Inventory
# RetailPulse Data Pipeline
# ============================================================
# Reads inventory snapshot CSV from Bronze layer
# Produces one Delta table in Silver:
#   - retailpulse.silver.inventory
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *

# ------------------------------------------------------------
# CELL 1: Read Bronze inventory data
# ------------------------------------------------------------

bronze_path = "abfss://bronze@retailpulsedatalake.dfs.core.windows.net/inventory/inventory.csv"

df_bronze = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(bronze_path)

print(f"Row count: {df_bronze.count()}")
display(df_bronze)

# ------------------------------------------------------------
# CELL 2: Clean, transform and add computed columns
# ------------------------------------------------------------

df_silver = df_bronze \
    .withColumn("delivery_date",
        F.to_date(F.col("delivery_date"))
    ) \
    .withColumn("last_restocked_date",
        F.to_date(F.col("last_restocked_date"))
    ) \
    .withColumn("is_low_stock",
        F.when(
            F.col("current_stock") < F.col("reorder_threshold"), 
            True
        ).otherwise(False)
    ) \
    .withColumn("stock_gap",
        F.when(
            F.col("current_stock") < F.col("reorder_threshold"),
            F.col("reorder_threshold") - F.col("current_stock")
        ).otherwise(0)
    )

display(df_silver.select(
    "purchase_order_id",
    "product_name", 
    "current_stock",
    "reorder_threshold",
    "is_low_stock",
    "stock_gap"
))

# ------------------------------------------------------------
# CELL 3: Write to Silver Delta table
# ------------------------------------------------------------

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailpulse.silver.inventory")

print("Inventory Silver table written successfully")


Row count: 10


id,purchase_order_id,supplier_id,store_id,product_id,product_name,units_ordered,units_received,unit_cost,total_cost,current_stock,reorder_threshold,warehouse_location,delivery_date,delivery_status,delivery_cost,last_restocked_date
1,PO_001,SUP_001,STORE_001,SKU_001,Maize Meal 10kg,500,500,8.5,4250.0,320,100,Harare Central,2026-05-20,delivered,150.0,2026-05-20
2,PO_002,SUP_001,STORE_002,SKU_001,Maize Meal 10kg,400,380,8.5,3230.0,95,100,Bulawayo West,2026-05-21,delivered,120.0,2026-05-21
3,PO_003,SUP_002,STORE_001,SKU_002,Cooking Oil 2L,300,300,5.2,1560.0,210,80,Harare Central,2026-05-22,delivered,90.0,2026-05-22
4,PO_004,SUP_003,STORE_003,SKU_003,Rice 5kg,600,600,6.3,3780.0,450,120,Mutare East,2026-05-23,delivered,180.0,2026-05-23
5,PO_005,SUP_003,STORE_002,SKU_003,Rice 5kg,200,150,6.3,945.0,60,80,Bulawayo West,2026-05-24,delivered,75.0,2026-05-24
6,PO_006,SUP_004,STORE_001,SKU_006,Milk 1L,1000,1000,1.2,1200.0,45,150,Harare Central,2026-05-25,delivered,50.0,2026-05-25
7,PO_007,SUP_001,STORE_003,SKU_004,Sugar 2kg,350,350,3.1,1085.0,280,90,Mutare East,2026-05-26,delivered,95.0,2026-05-26
8,PO_008,SUP_002,STORE_001,SKU_005,Bread Loaf,800,790,1.5,1185.0,310,200,Harare Central,2026-05-27,delivered,60.0,2026-05-27
9,PO_009,SUP_003,STORE_002,SKU_007,Eggs 6 pack,400,400,2.2,880.0,30,100,Bulawayo West,2026-05-28,in_transit,110.0,2026-04-15
10,PO_010,SUP_004,STORE_003,SKU_008,Chicken 1kg,250,0,4.5,0.0,15,50,Mutare East,2026-05-29,ordered,85.0,2026-04-20


purchase_order_id,product_name,current_stock,reorder_threshold,is_low_stock,stock_gap
PO_001,Maize Meal 10kg,320,100,false,0
PO_002,Maize Meal 10kg,95,100,true,5
PO_003,Cooking Oil 2L,210,80,false,0
PO_004,Rice 5kg,450,120,false,0
PO_005,Rice 5kg,60,80,true,20
PO_006,Milk 1L,45,150,true,105
PO_007,Sugar 2kg,280,90,false,0
PO_008,Bread Loaf,310,200,false,0
PO_009,Eggs 6 pack,30,100,true,70
PO_010,Chicken 1kg,15,50,true,35


Inventory Silver table written successfully
